# BƯỚC 2.1: TIỀN XỬ LÝ, LÀM SẠCH DỮ LIỆU & TRÍCH XUẤT ĐẶC TRƯNG (DATA CLEANING & FEATURE EXTRACTION)

**Môn học:** Khai phá dữ liệu (ITEC3417) — Đồ án Thực tế Tuyển dụng CNTT & Khoa học Dữ liệu tại Việt Nam  
**Đầu vào:** `Giai Đoạn 1/data/raw_jobs_data.csv` (1.473 bản ghi thô từ CareerViet, TopCV, VietnamWorks) & `data/reference/skill_dictionary.csv` (73 kỹ năng thuộc 7 nhóm kỹ năng)  
**Đầu ra chuẩn:** `data/clean/clean_data.csv` (20 cột cố định + 73 cột nhị phân `skill_<nhóm>_<tên>` = 93 cột)

---

### Ánh xạ Quy trình Thực hiện (Bám sát Giáo trình `TH01–TH02` & Mở rộng cho Dữ liệu Thực tế)

| Thứ tự | Hạng mục xử lý | Kỹ thuật cốt lõi từ `TH01` & `TH02` | Phần mở rộng cho dữ liệu tuyển dụng thực tế |
| :---: | :--- | :--- | :--- |
| **1** | **Data Profiling sơ bộ** | `df.shape`, `df.info()`, `df.head()`, `df.describe(include='all')`, đếm khuyết trước xử lý | Kiểm tra thêm chuỗi rỗng `""`, phân bố tin theo sàn & từ khóa tìm kiếm |
| **2** | **Lọc tin lạc ngành & Khử trùng** | Đếm `.duplicated().sum()` và `drop_duplicates()` | Lọc 3 lớp: bỏ cụm lỗi tìm kiếm `topcv + Tableau`, lọc tin ngoài ngành kết hợp Tiêu đề + Từ điển kỹ năng, khử trùng đa nguồn theo `(title, company)` ưu tiên bản ghi đầy đủ nhất, xuất Nhật ký làm sạch |
| **3** | **Chuẩn hóa Mức lương (Regex)** | Chiến lược xử lý khuyết (không gán `0`/Mean làm méo phân phối), lọc giá trị phi lý (`$ 1-1`), rời rạc hóa `pd.cut(..., include_lowest=True)` | Hàm Regex `re.VERBOSE` giải mã đa định dạng, phân biệt dấu `.`/`,` hàng nghìn vs thập phân, quy đổi USD theo tỷ giá NHNN ngày `19/11/2026` (`25.646 VNĐ/USD`), chia `/năm` cho `12`, xuất bảng đối soát mẫu |
| **4** | **Chuẩn hóa Địa điểm** | `.str.strip()`, ánh xạ từ điển `.replace({...})` | Cắt địa chỉ chi tiết sau `\n` (CareerViet), bỏ hậu tố `(mới)` (TopCV), lọc tên Quận/Huyện đi kèm Tỉnh/Thành (VietnamWorks), quét `Remote/WFH` trên địa điểm & tiêu đề |
| **5** | **Kinh nghiệm & Cấp bậc** | Rời rạc hóa `pd.cut()`, phân biệt mã hóa **Ordinal** vs **One-Hot** | Bóc số năm kinh nghiệm tối thiểu `Exp_Min_Years`, suy luận `Level` từ tiêu đề (chặn bẫy từ `"chính"` và `"chuyên viên quản lý..."`) kết hợp quy tắc fallback từ kinh nghiệm |
| **6** | **Văn bản, Stop Words & Ma trận Kỹ năng** | Biến đổi đặc trưng dạng nhị phân One-Hot (`int` 0/1) | Gộp `title + description + requirements` (bỏ `skills_raw` của TopCV), gỡ HTML (`BeautifulSoup`), xóa dấu câu trang trí thừa nhưng bảo toàn ký tự kỹ thuật (`+`, `#`, `.`, `/`, `-`), bỏ **Stop Words Việt/Anh**, khớp 73 regex có ranh giới từ |
| **7** | **Kiểm định Toàn vẹn (`assert`)** | Kiểm tra điều kiện logic đầu ra bằng `assert` | Kiểm định 20 cột cố định + 73 cột `skill_*`, bất biến `Luong_Min <= Luong_Max`, chống rò rỉ dữ liệu và xuất `clean_data.csv` |

In [1]:
# 1. Nạp thư viện trong phạm vi requirement.txt và cấu hình dự án
import csv
import html
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

# Đảm bảo import được config.py dù mở Notebook tại thư mục gốc hay bên trong `Giai Đoạn 2/`
current_dir = Path.cwd().resolve()
for candidate_dir in (
    current_dir,
    current_dir.parent,
    current_dir / "giai_doan_2.1",
):
    if (candidate_dir / "config.py").exists() and str(candidate_dir) not in sys.path:
        sys.path.insert(0, str(candidate_dir))
        break

import config

RANDOM_STATE: int = 42
csv.field_size_limit(10**7)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)

config.ensure_directories()

assert config.RAW_DATA_PATH.exists(), f"Không tìm thấy dữ liệu thô tại: {config.RAW_DATA_PATH}"
assert config.SKILL_DICT_PATH.exists(), f"Không tìm thấy từ điển kỹ năng tại: {config.SKILL_DICT_PATH}"

print("=== CẤU HÌNH PIPELINE BƯỚC 2.1 ===")
print(f"Thư mục gốc (PROJECT_ROOT) : {config.PROJECT_ROOT}")
print(f"Đường dẫn dữ liệu thô      : {config.RAW_DATA_REL_PATH} (exists={config.RAW_DATA_PATH.exists()})")
print(f"Đường dẫn từ điển kỹ năng  : {config.SKILL_DICT_REL_PATH} (exists={config.SKILL_DICT_PATH.exists()})")
print(f"Đường dẫn dữ liệu sạch đích: {config.CLEAN_DATA_REL_PATH}")
print(f"Tỷ giá USD cố định         : 1 USD = {config.USD_TO_VND:,} VNĐ ({config.USD_RATE_SOURCE}, ngày {config.USD_RATE_DATE})")

=== CẤU HÌNH PIPELINE BƯỚC 2.1 ===
Thư mục gốc (PROJECT_ROOT) : C:\Users\letrungtanh\Videos\NVIDIA\Gemini\Khai Phá Dữ Liệu\giai_doan_2.1
Đường dẫn dữ liệu thô      : Giai Đoạn 1/data/raw_jobs_data.csv (exists=True)
Đường dẫn từ điển kỹ năng  : data/reference/skill_dictionary.csv (exists=True)
Đường dẫn dữ liệu sạch đích: data/clean/clean_data.csv
Tỷ giá USD cố định         : 1 USD = 25,646 VNĐ (Tỷ giá trung tâm — Ngân hàng Nhà nước Việt Nam (NHNN), ngày 2026-11-19)


---
## 1. Khám phá sơ bộ dữ liệu thô (Data Profiling — Bám sát `TH01` & `TH02` Bước 2)

**Mục tiêu & Lý do thực hiện:**
- Khảo sát cấu trúc tổng thể của tập dữ liệu thô (`shape`, `info`, `head`, `describe(include='all')`) trước khi can thiệp làm sạch.
- Thống kê chính xác số lượng và tỷ lệ khuyết thiếu ban đầu trên từng cột (tính cả `NaN` lẫn chuỗi rỗng `""`) để làm mốc cơ sở đối chiếu.
- Kiểm tra đặc thù dữ liệu của 3 sàn tuyển dụng (`careerviet`, `topcv`, `vietnamworks`) nhằm thiết kế quy tắc tiền xử lý phù hợp.

In [2]:
# Đọc dữ liệu thô với mã hóa chuẩn từ config.py
df_raw = pd.read_csv(config.RAW_DATA_PATH, encoding=config.CSV_ENCODING)

print(f"Kích thước dữ liệu thô ban đầu: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")
print("\n--- 1. Thông tin kiểu dữ liệu (df_raw.info) ---")
df_raw.info()

# Thống kê số lượng và tỷ lệ khuyết thiếu (ép sang object trước khi fillna('') để tương thích Pandas 3.x)
missing_rows = []
for col in df_raw.columns:
    na_cnt = int(df_raw[col].isna().sum())
    col_as_str = df_raw[col].astype("object").fillna("").astype(str).str.strip()
    empty_cnt = int(col_as_str.eq("").sum())
    missing_rows.append({
        "Tên cột": col,
        "Kiểu dữ liệu": str(df_raw[col].dtype),
        "Số ô NaN": na_cnt,
        "Số ô rỗng/khuyết thực tế": empty_cnt,
        "Tỷ lệ khuyết (%)": round(empty_cnt / len(df_raw) * 100, 2),
        "Số giá trị duy nhất": int(df_raw[col].nunique(dropna=True)),
    })

df_missing_profile = pd.DataFrame(missing_rows)
print("\n--- 2. Bảng khảo sát khuyết thiếu & độ đa dạng trên 17 cột dữ liệu thô ---")
print(df_missing_profile.to_string(index=False))

print("\n--- 3. Thống kê mô tả tổng quát (df_raw.describe(include='all')) trên các cột chính ---")
summary_profile_cols = ["site", "title", "company", "location", "salary_raw", "experience_raw", "keyword_search"]
print(df_raw[summary_profile_cols].describe(include="all").to_string())

print("\n--- 4. Phân bố số tin theo nền tảng tuyển dụng (site) ---")
print(df_raw["site"].value_counts().to_frame(name="Số lượng tin"))

print("\n--- 5. Xem trước 3 bản ghi đầu tiên (df_raw.head(3)) ---")
preview_cols = ["job_id", "site", "title", "company", "location", "salary_raw", "experience_raw", "keyword_search"]
print(df_raw[preview_cols].head(3).to_string(index=False))

Kích thước dữ liệu thô ban đầu: 1,473 dòng x 17 cột

--- 1. Thông tin kiểu dữ liệu (df_raw.info) ---
<class 'pandas.DataFrame'>
RangeIndex: 1473 entries, 0 to 1472
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   job_id               1473 non-null   str    
 1   site                 1473 non-null   str    
 2   URL_Goc              1473 non-null   str    
 3   Timestamp            1473 non-null   str    
 4   title                1473 non-null   str    
 5   company              1473 non-null   str    
 6   location             1468 non-null   str    
 7   salary_raw           1468 non-null   str    
 8   experience_raw       1421 non-null   str    
 9   description          1472 non-null   str    
 10  requirements         1468 non-null   str    
 11  benefits             1393 non-null   str    
 12  skills_raw           718 non-null    str    
 13  work_model_raw       0 non-null      float64
 14

**Nhận xét kết quả Data Profiling ban đầu:**
- Các cột truy vết cốt lõi (`job_id`, `site`, `URL_Goc`, `Timestamp`, `title`, `company`, `keyword_search`) đầy đủ 100%; các cột `location`, `salary_raw`, `requirements` chỉ khuyết `0,34%` (5 dòng) và `experience_raw` khuyết `3,53%` (52 dòng).
- Cột tùy chọn `work_model_raw` khuyết `100%` do các sàn không tách thẻ hình thức làm việc đồng nhất → cần quét từ khóa `Remote / WFH` từ địa điểm và tiêu đề.
- Cột `skills_raw` khuyết `51,26%` và trên TopCV chứa chuỗi thẻ SEO rất dài → để đảm bảo tính nhất quán giữa 3 sàn, ta trích xuất kỹ năng trực tiếp từ văn bản gộp `title + description + requirements`.

---
## 2. Lọc tin lạc ngành & Khử trùng lặp đa nguồn (`TH02` Bước 3 + Mở rộng)

**Mục tiêu & Thiết kế 3 lớp lọc khách quan:**
1. **Lớp 1 — Loại cụm lỗi tìm kiếm `topcv + keyword_search == 'Tableau'`:**  
   Truy vấn từ khóa `Tableau` trên TopCV bị lỗi cơ chế tìm kiếm của sàn, trả về 90/91 tin ngoài ngành (*Giáo viên, Bác sĩ, Kế toán, HSE...*).
2. **Lớp 2 — Lọc tin lạc ngành ở các sàn còn lại (Kết hợp Tiêu đề & Từ điển kỹ năng):**  
   - Loại các tin có tiêu đề thuộc nghề ngoài ngành (*Kế toán / Accounting, Kiểm toán, Y tế, Giáo dục, Sales / Vận hành kinh doanh thuần, Marketing / Trade Marketing, Phân tích đầu tư / Trái phiếu / Thẩm định tín dụng Underwriting, Kỹ sư cơ khí / xây dựng / chất lượng phần cứng nhà máy...*) mà trên tiêu đề **không** có từ khóa công nghệ/dữ liệu cốt lõi (`Data`, `BI`, `Power BI`, `Tableau`, `SQL`, `Python`, `AI`, `BA`, `Business Analyst`, `Mô hình rủi ro / Risk Modeling`, `Phân tích định lượng`, `MIS`, `IT`, `Phần mềm`).
   - **Bảo toàn phân khúc Data Science Ngân hàng:** Giữ lại đầy đủ các vị trí *Xây dựng & Kiểm định Mô hình Rủi ro (Risk Modeling / Risk Analytics / Credit Scoring / Quantitative Analyst)* và *Sales BI / Product Owner* sử dụng `SQL`, `Python`, `R`, `Machine Learning`, `Power BI`.
   - Loại các tin không có từ khóa vị trí CNTT/Dữ liệu/BA trên tiêu đề và chỉ khớp duy nhất `Excel` văn phòng hoặc không khớp bất kỳ kỹ năng chuyên môn nào (`0` kỹ năng cứng).
3. **Lớp 3 — Khử trùng lặp đa nguồn (`job_id`, `URL_Goc` và cặp `(title, company)` chuẩn hóa):**  
   Sau khi kiểm tra trùng khóa kỹ thuật (`job_id`, `URL_Goc`), thực hiện khử trùng theo cặp `(title, company)` đã chuẩn hóa chữ thường/khoảng trắng để loại tin đăng lặp hoặc đăng chéo giữa 3 sàn (ưu tiên giữ bản ghi có lương dạng số, có kinh nghiệm và có mô tả dài nhất).

In [3]:
# [Mở rộng ngoài TH01-TH06] Cell 2A: Định nghĩa biểu thức Regex và hàm lọc tin lạc ngành
df_skill_dict = pd.read_csv(config.SKILL_DICT_PATH, encoding=config.CSV_ENCODING)

COMPILED_SKILLS: list[tuple[str, str, str, re.Pattern]] = [
    (
        row["category"],
        row["skill_name"],
        row["column_name"],
        re.compile(row["regex_pattern"], 0 if int(row["case_sensitive"]) == 1 else re.IGNORECASE),
    )
    for _, row in df_skill_dict.iterrows()
]

CORE_TECH_DOMAIN_SKILLS: set[str] = {
    "Power BI", "Tableau", "Looker / Looker Studio", "Power Query / Power Platform",
    "BPMN / UML", "Jira", "Confluence", "Figma", "Diagramming (Draw.io/Visio/Miro)",
    "Machine Learning", "LLM / GenAI / RAG / Agent", "ETL / Data Pipeline",
    "Data Modeling", "Data Governance / Quality", "BA Documentation (BRD/SRS/User Story)",
    "Big Data", "Computer Vision", "Deep Learning / NLP", "Data Visualization / Dashboard",
    "Statistics / A/B Testing",
}

OFF_DOMAIN_TITLE_PATTERN = re.compile(
    r'''
    (?:
        giáo\s*viên | teacher | giảng\s*viên\s*tiếng |                  # Giáo dục / Giảng dạy ngoại ngữ
        bác\s*sĩ | dược\s*sĩ | y\s*tá | điều\s*dưỡng |                  # Y tế / Dược
        kế\s*toán | accountant | accounting |                           # Kế toán (Việt & Anh)
        kiểm\s*toán\s*(?:viên\s*)?(?:nội\s*bộ|tài\s*chính) |            # Kiểm toán thuần
        nhân\s*viên\s*kinh\s*doanh | chuyên\s*viên\s*kinh\s*doanh | trưởng\s*phòng\s*kinh\s*doanh |
        phát\s*triển\s*kinh\s*doanh(?!\s*số) | phòng\s*kinh\s*doanh |   # Kinh doanh / Sales thương mại thuần
        sales\s*(?:assistant|planning|operations|admin|support|executive|manager) |
        quản\s*lý\s*vận\s*hành\s*(?:khối\s*)?kinh\s*doanh |
        tư\s*vấn\s*tuyển\s*sinh | tư\s*vấn\s*bán\s*hàng | telesales | bất\s*động\s*sản |
        (?:chiến\s*lược|phân\s*tích|thực\s*tập\s*sinh\s*phân\s*tích)\s*đầu\s*tư |
        investment\s*analyst | quan\s*hệ\s*nhà\s*đầu\s*tư | investor\s*relations |
        đầu\s*tư\s*(?:trái\s*phiếu|và\s*phát\s*hành) | covered\s*warrants | derivative\s*products |
        underwriting | thẩm\s*định\s*(?:tín\s*dụng|hồ\s*sơ|bảo\s*hiểm) |
        kiểm\s*soát\s*rủi\s*ro\s*công\s*nợ | quản\s*trị\s*nợ |
        (?:nhân\s*viên|chuyên\s*viên|trưởng\s*bộ\s*phận|quản\s*lý|vice\s*head\s*of)\s*(?:trade\s*|digital\s*|growth\s*)?marketing |
        (?:trade|digital|growth)\s*marketing\s*(?:intern|executive|manager|specialist|lead) |
        marketing\s*(?:sản\s*phẩm|dịch\s*vụ|lead|manager|executive) | brand\s*manager |
        kỹ\s*sư\s*(?:cơ\s*khí|xây\s*dựng|điện\s*(?:lạnh|công\s*nghiệp)|hóa\s*học|nông\s*nghiệp) |
        mechanical\s*engineer | civil\s*engineer | microbiology | vi\s*sinh |
        failure\s*analysis | faca\s*engineer | customer-facing\s*quality |
        quản\s*lý\s*dự\s*án\s*nhà\s*máy | quản\s*lý\s*phát\s*hành\s*phim |
        lái\s*xe | tài\s*xế | bảo\s*vệ | tạp\s*vụ | thủ\s*kho | nhân\s*viên\s*kho |
        hành\s*chính\s*nhân\s*sự | tuyển\s*dụng | hr\s*(?:executive|manager|specialist) | c&b |
        chăm\s*sóc\s*khách\s*hàng | cskh | customer\s*service | lễ\s*tân | receptionist |
        thu\s*mua | mua\s*hàng | purchasing | buyer | procurement | xuất\s*nhập\s*khẩu | logistics\s*staff |
        luật\s*sư | pháp\s*chế | legal | trợ\s*lý\s*giám\s*đốc | thư\s*ký | secretary |
        biên\s*phiên\s*dịch | interpreter | translator |
        content\s*creator | tiktok | livestream | graphic\s*designer | thiết\s*kế\s*đồ\s*họa
    )
    ''',
    re.IGNORECASE | re.VERBOSE,
)

IN_DOMAIN_TITLE_PATTERN = re.compile(
    r'''
    \b(?:
        data | dữ\s*liệu | bi | business\s*intelligence | power\s*bi | tableau | sql | python |
        ai | artificial\s*intelligence | trí\s*tuệ\s*nhân\s*tạo | ml | machine\s*learning |
        business\s*analyst | ba | phân\s*tích\s*(?:dữ\s*liệu|nghiệp\s*vụ|kinh\s*doanh|hệ\s*thống|định\s*lượng) |
        mô\s*hình\s*rủi\s*ro | risk\s*(?:model|modeling|analytics) | chấm\s*điểm\s*tín\s*dụng |
        analytics | scientist | khoa\s*học\s*dữ\s*liệu |
        developer | lập\s*trình | software | phần\s*mềm | it | cntt | công\s*nghệ\s*thông\s*tin |
        cloud | devops | mlops | dataops | erp | sap | odoo | crm | mis |
        system | hệ\s*thống | database | csdl | dba |
        product\s*owner | product\s*manager | qa | qc | tester | kiểm\s*thử
    )\b
    ''',
    re.IGNORECASE | re.VERBOSE,
)

TITLE_TECH_OVERRIDE_PATTERN = re.compile(
    r"\b(?:data|dữ\s*liệu|bi|business\s*intelligence|power\s*bi|tableau|sql|python|ai|ba|business\s*analyst|"
    r"mô\s*hình\s*rủi\s*ro|risk\s*model(?:ing)?|risk\s*analytics|phân\s*tích\s*định\s*lượng|mis|it|cntt|phần\s*mềm|software)\b",
    re.IGNORECASE,
)


def strip_html_and_whitespace(raw_text: object) -> str:
    '''Gỡ thẻ HTML, giải mã HTML entities và gom khoảng trắng thừa về 1 dấu cách.'''
    if pd.isna(raw_text) or not str(raw_text).strip():
        return ""
    text = str(raw_text)
    if "<" in text and ">" in text:
        text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = html.unescape(text)
    return re.sub(r"\s+", " ", text).strip()


def evaluate_domain_relevance(title: str, combined_text: str) -> tuple[bool, str]:
    '''Đánh giá tin tuyển dụng có thuộc diện lạc ngành hay không và trả về (is_off_domain, lý_do).'''
    hard_hits, core_hits = 0, 0
    for cat, name, _, rx in COMPILED_SKILLS:
        if rx.search(combined_text):
            if cat != "soft":
                hard_hits += 1
            if cat in ("lang", "framework", "cloud", "db") or name in CORE_TECH_DOMAIN_SKILLS:
                core_hits += 1

    if OFF_DOMAIN_TITLE_PATTERN.search(title) and not TITLE_TECH_OVERRIDE_PATTERN.search(title):
        return (True, "Tiêu đề thuộc nhóm nghề ngoài CNTT/Dữ liệu/BA")
    if core_hits == 0 and not IN_DOMAIN_TITLE_PATTERN.search(title):
        return (True, "Không khớp kỹ năng cốt lõi (ngoài Excel) và tiêu đề không thuộc vị trí CNTT/Dữ liệu/BA")
    if hard_hits == 0 and not TITLE_TECH_OVERRIDE_PATTERN.search(title):
        return (True, "Nội dung JD không chứa bất kỳ kỹ năng chuyên môn nào trong từ điển")
    return (False, "Hợp lệ (Đúng phân khúc CNTT/Dữ liệu/BA)")

In [4]:
# [Mở rộng ngoài TH01-TH06] Cell 2B: Thực thi 3 lớp lọc, khử trùng lặp và xuất Nhật ký làm sạch
audit_log: list[dict[str, object]] = []
n_initial = len(df_raw)
audit_log.append({
    "Bước": "0. Dữ liệu thô ban đầu (raw_jobs_data.csv)",
    "Số dòng đầu vào": n_initial,
    "Số dòng bị loại": 0,
    "Số dòng còn lại": n_initial,
    "Tỷ lệ giữ lại (%)": 100.0,
    "Mô tả / Lý do xử lý": "Tổng số bản ghi thu thập từ CareerViet (755), TopCV (414), VietnamWorks (304)",
})

# Lớp 1: Loại cụm lỗi tìm kiếm topcv + keyword_search == 'Tableau'
mask_topcv_tableau = (
    df_raw["site"].astype(str).str.strip().str.lower().eq("topcv")
    & df_raw["keyword_search"].fillna("").astype(str).str.strip().eq("Tableau")
)
n_removed_topcv_tab = int(mask_topcv_tableau.sum())
df_step1 = df_raw.loc[~mask_topcv_tableau].copy()
audit_log.append({
    "Bước": "1. Loại cụm lỗi search TopCV (keyword_search = 'Tableau')",
    "Số dòng đầu vào": n_initial,
    "Số dòng bị loại": n_removed_topcv_tab,
    "Số dòng còn lại": len(df_step1),
    "Tỷ lệ giữ lại (%)": round(len(df_step1) / n_initial * 100, 2),
    "Mô tả / Lý do xử lý": "Cơ chế search từ khóa 'Tableau' trên TopCV trả về 90/91 tin ngoài ngành (giáo viên, bác sĩ, kế toán...)",
})

# Chuẩn bị văn bản gộp (title + description + requirements)
df_step1["title"] = df_step1["title"].fillna("").astype(str).map(strip_html_and_whitespace)
df_step1["company"] = df_step1["company"].fillna("").astype(str).map(strip_html_and_whitespace)
df_step1["_clean_desc"] = df_step1["description"].map(strip_html_and_whitespace)
df_step1["_clean_req"] = df_step1["requirements"].map(strip_html_and_whitespace)
df_step1["_combined_text"] = (
    df_step1["title"] + " " + df_step1["_clean_desc"] + " " + df_step1["_clean_req"]
).str.strip()

# Lớp 2: Lọc tin lạc ngành kết hợp Tiêu đề & Từ điển kỹ năng
domain_eval = [evaluate_domain_relevance(t, txt) for t, txt in zip(df_step1["title"], df_step1["_combined_text"])]
df_step1["_is_off_domain"] = [item[0] for item in domain_eval]
df_step1["_filter_reason"] = [item[1] for item in domain_eval]

# Xuất bảng mẫu kiểm toán các tin bị loại ở Lớp 2 ra file CSV để đối soát minh bạch
df_offdomain_review = df_step1.loc[
    df_step1["_is_off_domain"],
    ["job_id", "site", "keyword_search", "title", "company", "_filter_reason"],
].rename(columns={"_filter_reason": "reason"})
offdomain_sample_path = config.TABLES_DIR / "offdomain_review_sample.csv"
df_offdomain_review.to_csv(offdomain_sample_path, index=False, encoding=config.CSV_ENCODING)

n_removed_off_domain = int(df_step1["_is_off_domain"].sum())
df_step2 = df_step1.loc[~df_step1["_is_off_domain"]].copy()
audit_log.append({
    "Bước": "2. Lọc tin lạc ngành (Tiêu đề + Từ điển kỹ năng)",
    "Số dòng đầu vào": len(df_step1),
    "Số dòng bị loại": n_removed_off_domain,
    "Số dòng còn lại": len(df_step2),
    "Tỷ lệ giữ lại (%)": round(len(df_step2) / n_initial * 100, 2),
    "Mô tả / Lý do xử lý": "Loại tin kế toán, y tế, cơ khí, marketing, sales, phân tích đầu tư và tin không chứa kỹ năng chuyên môn",
})

# Lớp 3A: Kiểm tra trùng lặp khóa kỹ thuật (site, job_id) và URL_Goc (bám sát TH02 Bước 3)
n_before_3a = len(df_step2)
df_step2 = (
    df_step2.drop_duplicates(subset=["site", "job_id"], keep="first")
    .drop_duplicates(subset=["URL_Goc"], keep="first")
    .copy()
)
n_dup_id_url = n_before_3a - len(df_step2)
audit_log.append({
    "Bước": "3A. Khử trùng lặp khóa kỹ thuật (job_id / URL_Goc)",
    "Số dòng đầu vào": n_before_3a,
    "Số dòng bị loại": n_dup_id_url,
    "Số dòng còn lại": len(df_step2),
    "Tỷ lệ giữ lại (%)": round(len(df_step2) / n_initial * 100, 2),
    "Mô tả / Lý do xử lý": "Kiểm tra trùng mã tin và đường dẫn gốc từ crawler",
})

# Lớp 3B: Khử trùng lặp theo cặp (title, company) đã chuẩn hóa, ưu tiên bản ghi đầy đủ thông tin hơn
df_step2["_norm_title"] = df_step2["title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
df_step2["_norm_company"] = df_step2["company"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

salary_text_series = df_step2["salary_raw"].fillna("").astype(str)
has_numeric_salary = salary_text_series.str.contains(r"\d", regex=True) & ~salary_text_series.str.contains(r"\$\s*1\s*-\s*1\b", regex=True)
has_exp_info = df_step2["experience_raw"].fillna("").astype(str).str.strip().ne("")

df_step2["_quality_score"] = (
    has_numeric_salary.astype(int) * 10_000_000
    + has_exp_info.astype(int) * 1_000_000
    + df_step2["_combined_text"].str.len()
)

n_before_tc_dedup = len(df_step2)
df_clean = (
    df_step2.sort_values(by="_quality_score", ascending=False, kind="mergesort")
    .drop_duplicates(subset=["_norm_title", "_norm_company"], keep="first")
    .sort_index()
    .copy()
)
n_dup_tc = n_before_tc_dedup - len(df_clean)

audit_log.append({
    "Bước": "3B. Khử trùng lặp đa nguồn theo cặp (title, company)",
    "Số dòng đầu vào": n_before_tc_dedup,
    "Số dòng bị loại": n_dup_tc,
    "Số dòng còn lại": len(df_clean),
    "Tỷ lệ giữ lại (%)": round(len(df_clean) / n_initial * 100, 2),
    "Mô tả / Lý do xử lý": "Loại tin đăng lặp lại hoặc đăng chéo giữa 3 sàn; ưu tiên giữ bản ghi đủ lương, kinh nghiệm và JD dài nhất",
})

df_audit_log = pd.DataFrame(audit_log)
audit_log_path = config.TABLES_DIR / "cleaning_audit_log.csv"
df_audit_log.to_csv(audit_log_path, index=False, encoding=config.CSV_ENCODING)

print("=== BẢNG NHẬT KÝ LÀM SẠCH DỮ LIỆU (CLEANING AUDIT LOG) ===")
print(df_audit_log.to_string(index=False))
print(f"\nĐã xuất nhật ký làm sạch tại : {audit_log_path}")
print(f"Đã xuất danh sách tin bị loại: {offdomain_sample_path} ({len(df_offdomain_review)} dòng)")
print("\n--- Phân bố số tin sạch theo sàn tuyển dụng ---")
print(df_clean["site"].value_counts().to_frame(name="Số tin sạch"))

=== BẢNG NHẬT KÝ LÀM SẠCH DỮ LIỆU (CLEANING AUDIT LOG) ===
                                                     Bước  Số dòng đầu vào  Số dòng bị loại  Số dòng còn lại  Tỷ lệ giữ lại (%)                                                                                       Mô tả / Lý do xử lý
               0. Dữ liệu thô ban đầu (raw_jobs_data.csv)             1473                0             1473             100.00                             Tổng số bản ghi thu thập từ CareerViet (755), TopCV (414), VietnamWorks (304)
1. Loại cụm lỗi search TopCV (keyword_search = 'Tableau')             1473               91             1382              93.82   Cơ chế search từ khóa 'Tableau' trên TopCV trả về 90/91 tin ngoài ngành (giáo viên, bác sĩ, kế toán...)
         2. Lọc tin lạc ngành (Tiêu đề + Từ điển kỹ năng)             1382              196             1186              80.52   Loại tin kế toán, y tế, cơ khí, marketing, sales, phân tích đầu tư và tin không chứa kỹ năng chuyên môn
     

---
## 3. Chuẩn hóa Mức lương bằng Regular Expression (`TH02` Bước 4, 6 + Mở rộng Regex `re.VERBOSE`)

**Mục tiêu & Nguyên tắc xử lý:**
- **Đơn vị chuẩn hóa duy nhất:** Quy đổi toàn bộ về **triệu VNĐ/tháng** trên 3 cột số thực `Luong_Min`, `Luong_Max`, `Luong_Mid` và cột trạng thái `salary_status` (`range`, `single_bound`, `negotiable`, `missing`).
- **Bảo toàn phân phối thật (không gán `0` hay `Mean/Median` cho lương "Thỏa thuận"):** Khoảng ~70% tin tuyển dụng CNTT ghi lương thỏa thuận. Điền `0` sẽ kéo sập trung bình xuống gần 0, còn điền `Median` cho 70% dữ liệu sẽ tạo đỉnh giả tạo khổng lồ làm sai lệch kiểm định ở Bước 2.2 và làm mô hình phân lớp ở Bước 2.5 học trên nhãn giả.
- **Xử lý dấu `.` và `,` đa sàn bằng Regex Lookahead:** Mọi dấu `.` hoặc `,` đứng trước đúng cụm 3 chữ số (`(?=\d{3}(?:[.,]\d{3})*(?:\D|$))`, ví dụ `1.000`, `2,500`, `18.000.000`) được nhận diện là phân cách hàng nghìn; dấu `,` đứng trước 1–2 chữ số (`15,5`) là dấu thập phân. Quy đổi USD theo tỷ giá trung tâm NHNN cố định ngày `19/11/2026` (`1 USD = 25.646 VNĐ`) và chia `12` nếu trả theo `/năm`.

In [5]:
# [Mở rộng ngoài TH01-TH06] Cell 3A: Định nghĩa biểu thức Regex (re.VERBOSE) và hàm chuẩn hóa mức lương
SALARY_NUMBER_REGEX = re.compile(
    r'''
    \d+                 # Phần nguyên ban đầu (ít nhất 1 chữ số)
    (?:                 # Nhóm lặp không bắt (non-capturing) cho phần nghìn hoặc thập phân
        [.,]            # Dấu chấm hoặc dấu phẩy
        \d+             # Các chữ số theo sau
    )*                  # Có thể lặp nhiều nhóm (vd: 18.000.000 hoặc 1,500.50)
    ''',
    re.VERBOSE,
)

THOUSANDS_SEP_REGEX = re.compile(
    r'''
    [.,]                # Dấu '.' hoặc ',' đóng vai trò phân cách hàng nghìn
    (?=                 # Positive lookahead: theo sau bắt buộc là cụm đúng 3 chữ số
        \d{3}           # Đúng 3 chữ số (vd: .000 hoặc ,500)
        (?:[.,]\d{3})*  # Có thể nối tiếp các cụm 3 chữ số khác (vd: .000.000)
        (?:\D|$)        # Kết thúc bằng ký tự không phải số hoặc cuối chuỗi
    )
    ''',
    re.VERBOSE,
)

PLACEHOLDER_SALARY_REGEX = re.compile(r"\$\s*1\s*-\s*1\b", re.IGNORECASE)
NEGOTIABLE_KEYWORDS: tuple[str, ...] = ("thỏa thuận", "thoả thuận", "thương lượng", "cạnh tranh", "negotiable")
UPPER_BOUND_KEYWORDS: tuple[str, ...] = ("tới", "lên đến", "up to", "dưới", "tối đa", "max")


def normalize_numeric_token(token: str, unit: str) -> float:
    '''Chuyển chuỗi số thô (vd '1.500', '15,5', '18,000,000') về đơn vị triệu VNĐ.'''
    cleaned = THOUSANDS_SEP_REGEX.sub("", token.strip())
    cleaned = cleaned.replace(",", ".")
    value = float(cleaned)

    if unit == "usd":
        return config.usd_to_million_vnd(value)
    if unit == "vnd_raw":
        return value / 1_000_000 if value >= 100_000 else value
    return value


def parse_salary_string(salary_raw: object) -> tuple[float, float, float, str]:
    '''Bóc tách chuỗi lương thô thành (Luong_Min, Luong_Max, Luong_Mid, salary_status).'''
    if pd.isna(salary_raw) or not str(salary_raw).strip():
        return (np.nan, np.nan, np.nan, "missing")

    raw_str = str(salary_raw).strip()
    low = raw_str.lower()

    if PLACEHOLDER_SALARY_REGEX.search(low):
        return (np.nan, np.nan, np.nan, "missing")
    if any(kw in low for kw in NEGOTIABLE_KEYWORDS):
        return (np.nan, np.nan, np.nan, "negotiable")

    is_yearly = any(kw in low for kw in ("/năm", "mỗi năm", "per year", "/year"))
    period_divisor = 12.0 if is_yearly else 1.0

    if "usd" in low or "$" in low:
        unit = "usd"
    elif any(kw in low for kw in ("triệu", "tr ", "tr/", "tr₫")) or re.search(r"\d+\s*(?:triệu|tr|m)\b", low):
        unit = "million_vnd"
    else:
        unit = "vnd_raw"

    raw_numbers = SALARY_NUMBER_REGEX.findall(low)
    if not raw_numbers:
        return (np.nan, np.nan, np.nan, "missing")

    values = [round(normalize_numeric_token(tok, unit) / period_divisor, 2) for tok in raw_numbers]

    if len(values) >= 2:
        luong_min, luong_max = min(values[0], values[1]), max(values[0], values[1])
        if luong_min <= 0 and luong_max <= 0:
            return (np.nan, np.nan, np.nan, "missing")
        return (luong_min, luong_max, round((luong_min + luong_max) / 2.0, 2), "range")

    single_val = values[0]
    if single_val <= 0:
        return (np.nan, np.nan, np.nan, "missing")
    if any(kw in low for kw in UPPER_BOUND_KEYWORDS):
        return (np.nan, single_val, np.nan, "single_bound")
    return (single_val, np.nan, np.nan, "single_bound")

In [6]:
# Cell 3B: Áp dụng chuẩn hóa lương, rời rạc hóa Salary_Bracket (pd.cut) và xuất bảng đối soát 16 mẫu
salary_parsed = df_clean["salary_raw"].map(parse_salary_string)
df_clean["Luong_Min"] = [item[0] for item in salary_parsed]
df_clean["Luong_Max"] = [item[1] for item in salary_parsed]
df_clean["Luong_Mid"] = [item[2] for item in salary_parsed]
df_clean["salary_status"] = [item[3] for item in salary_parsed]

# Rời rạc hóa Luong_Mid thành Salary_Bracket bằng pd.cut (bám sát TH02_TrichXuatDacTrung.pdf Bước 4)
df_clean["Salary_Bracket"] = pd.cut(
    df_clean["Luong_Mid"],
    bins=[0.0, 15.0, 30.0, np.inf],
    labels=["Duoi_15M", "15M_30M", "Tren_30M"],
    include_lowest=True,
).astype("object")

print("=== 1. PHÂN BỐ TRẠNG THÁI MỨC LƯƠNG (salary_status) ===")
sal_dist = df_clean["salary_status"].value_counts().to_frame(name="Số tin")
sal_dist["Tỷ lệ (%)"] = (sal_dist["Số tin"] / len(df_clean) * 100).round(2)
print(sal_dist)

print("\n=== 2. THỐNG KÊ MÔ TẢ CÁC CỘT LƯƠNG SỐ (Đơn vị: Triệu VNĐ/tháng) ===")
print(df_clean[["Luong_Min", "Luong_Max", "Luong_Mid"]].describe().round(2))

print("\n=== 3. PHÂN BỐ NHÃN KHOẢNG LƯƠNG (Salary_Bracket — trên các tin 'range') ===")
bracket_dist = df_clean["Salary_Bracket"].value_counts(dropna=False).to_frame(name="Số tin")
bracket_dist["Tỷ lệ trên toàn tập (%)"] = (bracket_dist["Số tin"] / len(df_clean) * 100).round(2)
print(bracket_dist)

# Lập bảng đối soát 16 mẫu chuỗi lương đa dạng từ cả 3 sàn để kiểm chứng bằng mắt & phục vụ vấn đáp
sample_audit_indices = []
for site_name in ("careerviet", "topcv", "vietnamworks"):
    for st_name in ("range", "single_bound", "negotiable", "missing"):
        sub = df_clean[(df_clean["site"] == site_name) & (df_clean["salary_status"] == st_name)]
        if not sub.empty:
            usd_sub = sub[sub["salary_raw"].fillna("").str.contains(r"USD|\$|/năm", case=False, regex=True)]
            if not usd_sub.empty:
                sample_audit_indices.append(usd_sub.index[0])
            sample_audit_indices.append(sub.index[-1])

sample_audit_indices = list(dict.fromkeys(sample_audit_indices))[:16]
df_salary_audit = df_clean.loc[
    sample_audit_indices,
    ["site", "salary_raw", "Luong_Min", "Luong_Max", "Luong_Mid", "salary_status", "Salary_Bracket"],
].copy()

salary_audit_path = config.TABLES_DIR / "salary_regex_audit.csv"
df_salary_audit.to_csv(salary_audit_path, index=False, encoding=config.CSV_ENCODING)

print("\n=== 4. BẢNG ĐỐI SOÁT 16 MẪU REGEX LƯƠNG ĐA SÀN (salary_regex_audit.csv) ===")
print(df_salary_audit.to_string(index=False))

=== 1. PHÂN BỐ TRẠNG THÁI MỨC LƯƠNG (salary_status) ===
               Số tin  Tỷ lệ (%)
salary_status                   
negotiable        802      69.20
range             304      26.23
single_bound       48       4.14
missing             5       0.43

=== 2. THỐNG KÊ MÔ TẢ CÁC CỘT LƯƠNG SỐ (Đơn vị: Triệu VNĐ/tháng) ===
       Luong_Min  Luong_Max  Luong_Mid
count     311.00     345.00     304.00
mean       19.28      30.91      24.79
std         9.43      15.11      11.67
min         1.00       0.50       1.50
25%        14.00      20.00      17.50
50%        18.00      30.00      22.50
75%        25.00      40.00      30.00
max        70.00     100.00      85.00

=== 3. PHÂN BỐ NHÃN KHOẢNG LƯƠNG (Salary_Bracket — trên các tin 'range') ===
                Số tin  Tỷ lệ trên toàn tập (%)
Salary_Bracket                                 
NaN                855                    73.77
15M_30M            179                    15.44
Tren_30M            71                     6.13
Duoi_15

---
## 4. Chuẩn hóa Địa điểm làm việc (`Location_Std` — `TH02` Bước 5 + Mở rộng)

**Mục tiêu & Quy tắc chuẩn hóa về 7 nhãn chuẩn (`Hà Nội`, `TP.HCM`, `Đà Nẵng`, `Remote`, `Đa địa điểm`, `Tỉnh khác`, `Không rõ`):**
- **Cắt địa chỉ chi tiết sau `\n` (CareerViet):** Tránh bắt nhầm tên tỉnh phụ nằm trong tên khu công nghiệp hoặc tên đường ở dòng thứ hai.
- **Loại bỏ hậu tố `(mới)` (TopCV) & tiền tố Quận/Huyện (VietnamWorks):** Tránh gán nhầm `"Quận 1, Hồ Chí Minh"` thành `Đa địa điểm`.
- **Quét từ khóa `Remote / Work From Home / WFH` trên `location` và `title` (không quét trong `description`):** Kiểm chứng thực nghiệm trên tập dữ liệu cho thấy 10/11 tin chứa từ `"remote"` hoặc `"wfh"` bên trong `description` là các ngữ cảnh kỹ thuật/phúc lợi phụ (*"provide remote support using TeamViewer"*, *"terraform remote backends"*, *"remote site protection"*, *"monthly happy friday WFH"*), không phải vị trí làm việc từ xa.

In [7]:
# [Mở rộng ngoài TH01-TH06] Cell 4A: Định nghĩa từ điển đồng nghĩa (TH02 Bước 5) và hàm chuẩn hóa địa điểm
REMOTE_PATTERN = re.compile(r"\b(?:remote|work\s*from\s*home|wfh|làm\s*việc\s*từ\s*xa)\b", re.IGNORECASE)
DISTRICT_PREFIX_PATTERN = re.compile(
    r"^(?:quận|huyện|thị\s*xã|phường|tp\.?\s*thủ\s*đức|thủ\s*đức)\b",
    re.IGNORECASE,
)
INNER_DISTRICT_NAMES_PATTERN = re.compile(
    r"\b(?:ba\s*đình|cầu\s*giấy|đống\s*đa|hai\s*bà\s*trưng|hoàn\s*kiếm|hoàng\s*mai|"
    r"long\s*biên|tây\s*hồ|thanh\s*xuân|nam\s*từ\s*liêm|bắc\s*từ\s*liêm|hà\s*đông|"
    r"thanh\s*trì|gia\s*lâm|đông\s*anh|sóc\s*sơn|tân\s*bình|bình\s*thạnh|phú\s*nhuận|"
    r"gò\s*vấp|tân\s*phú|bình\s*tân|nhà\s*bè|bình\s*chánh|hóc\s*môn|củ\s*chi|"
    r"hải\s*châu|thanh\s*khê|sơn\s*trà|ngũ\s*hành\s*sơn|liên\s*chiểu|cẩm\s*lệ)\b",
    re.IGNORECASE,
)

LOCATION_SYNONYM_MAP: dict[str, str] = {
    "hồ chí minh": "TP.HCM", "ho chi minh": "TP.HCM", "tp.hcm": "TP.HCM",
    "tphcm": "TP.HCM", "tp hcm": "TP.HCM", "hcm": "TP.HCM", "sài gòn": "TP.HCM",
    "hà nội": "Hà Nội", "ha noi": "Hà Nội", "hn": "Hà Nội",
    "đà nẵng": "Đà Nẵng", "da nang": "Đà Nẵng",
    "không xác định": "Không rõ", "khác": "Không rõ", "other": "Không rõ",
    "việt nam": "Không rõ", "vietnam": "Không rõ",
    "toàn quốc": "Đa địa điểm",
}


def normalize_location_record(location_raw: object, title: object) -> str:
    '''Chuẩn hóa địa điểm về 1 trong 7 nhãn: Hà Nội, TP.HCM, Đà Nẵng, Remote, Đa địa điểm, Tỉnh khác, Không rõ.'''
    loc_str = "" if pd.isna(location_raw) else str(location_raw).strip()
    title_str = "" if pd.isna(title) else str(title).strip()

    if REMOTE_PATTERN.search(f"{loc_str} {title_str}"):
        return "Remote"
    if not loc_str:
        return "Không rõ"

    first_line = loc_str.split("\n")[0].strip()
    cleaned_line = re.sub(r"\(mới\)", "", first_line, flags=re.IGNORECASE).strip()
    low_line = re.sub(r"\s+", " ", cleaned_line.lower())

    if low_line in LOCATION_SYNONYM_MAP:
        return LOCATION_SYNONYM_MAP[low_line]

    raw_parts = [p.strip() for p in re.split(r"[,;/|&]|\bvà\b|\band\b", low_line) if p.strip()]
    province_parts = [
        p for p in raw_parts
        if not DISTRICT_PREFIX_PATTERN.search(p) and not INNER_DISTRICT_NAMES_PATTERN.search(p)
        and p not in ("việt nam", "viet nam", "vietnam")
    ]

    has_hn = bool(re.search(r"hà\s*nội|ha\s*noi|\bhn\b", low_line))
    has_hcm = bool(re.search(r"hồ\s*chí\s*minh|ho\s*chi\s*minh|tp\.?\s*hcm|\bhcm\b|sài\s*gòn", low_line))
    has_dn = bool(re.search(r"đà\s*nẵng|da\s*nang", low_line))
    major_count = int(has_hn) + int(has_hcm) + int(has_dn)

    if major_count >= 2 or "toàn quốc" in low_line:
        return "Đa địa điểm"

    other_provinces = [
        p for p in province_parts
        if not re.search(r"hà\s*nội|ha\s*noi|\bhn\b|hồ\s*chí\s*minh|ho\s*chi\s*minh|tp\.?\s*hcm|\bhcm\b|đà\s*nẵng|da\s*nang", p)
    ]
    if (major_count == 1 and len(other_provinces) >= 1) or (major_count == 0 and len(other_provinces) >= 2):
        return "Đa địa điểm"

    if has_hn:
        return "Hà Nội"
    if has_hcm:
        return "TP.HCM"
    if has_dn:
        return "Đà Nẵng"
    if any(p in LOCATION_SYNONYM_MAP and LOCATION_SYNONYM_MAP[p] == "Không rõ" for p in province_parts):
        return "Không rõ"
    return "Tỉnh khác" 

In [8]:
# Cell 4B: Áp dụng chuẩn hóa địa điểm và hiển thị phân bố Location_Std
df_clean["Location_Std"] = [
    normalize_location_record(loc, ttl)
    for loc, ttl in zip(df_clean["location"], df_clean["title"])
]

print("=== PHÂN BỐ ĐỊA ĐIỂM ĐÃ CHUẨN HÓA (Location_Std) ===")
loc_dist = df_clean["Location_Std"].value_counts().to_frame(name="Số tin")
loc_dist["Tỷ lệ (%)"] = (loc_dist["Số tin"] / len(df_clean) * 100).round(2)
print(loc_dist)

print("\n--- Đối soát mẫu chuẩn hóa địa điểm từ dữ liệu thô -> Location_Std ---")
df_loc_preview = df_clean[["site", "location", "Location_Std"]].copy()
df_loc_preview["location_first_line"] = df_loc_preview["location"].fillna("").astype(str).str.split("\n").str[0]
df_loc_samples = df_loc_preview.drop_duplicates(subset=["Location_Std", "site"]).head(12)
print(df_loc_samples[["site", "location_first_line", "Location_Std"]].to_string(index=False))

=== PHÂN BỐ ĐỊA ĐIỂM ĐÃ CHUẨN HÓA (Location_Std) ===
              Số tin  Tỷ lệ (%)
Location_Std                   
Hà Nội           638      55.05
TP.HCM           361      31.15
Tỉnh khác        118      10.18
Đa địa điểm       22       1.90
Đà Nẵng           13       1.12
Remote             4       0.35
Không rõ           3       0.26

--- Đối soát mẫu chuẩn hóa địa điểm từ dữ liệu thô -> Location_Std ---
        site          location_first_line Location_Std
  careerviet                       Hà Nội       Hà Nội
  careerviet                  Hồ Chí Minh       TP.HCM
  careerviet                     Tây Ninh    Tỉnh khác
  careerviet Long An|Hồ Chí Minh|Đồng Nai  Đa địa điểm
  careerviet                      Đà Nẵng      Đà Nẵng
  careerviet                       Hà Nội       Remote
  careerviet                                  Không rõ
vietnamworks                       Hà Nội       Hà Nội
vietnamworks                     Hưng Yên    Tỉnh khác
vietnamworks                  Hồ Chí 

---
## 5. Chuẩn hóa Kinh nghiệm (`Exp_Min_Years`, `Exp_Group`) & Cấp bậc (`Level`) (`TH02` Trích xuất đặc trưng Bước 4–5)

**Mục tiêu & Quy tắc chuẩn hóa:**
1. **Số năm kinh nghiệm tối thiểu (`Exp_Min_Years`) & Nhóm kinh nghiệm (`Exp_Group`):**  
   Trích cận dưới số năm kinh nghiệm (`Không yêu cầu` → `0.0`; `Dưới 1 năm` hoặc `< 12 tháng` → `0.5`; `a - b năm` / `Trên a năm` → `a`), sau đó dùng `pd.cut(..., include_lowest=True)` chia thành 4 khoảng thứ bậc (`0 năm`, `1-2 năm`, `3-5 năm`, `>5 năm`) và gán `"Không rõ"` cho các dòng khuyết.
2. **Suy luận Cấp bậc (`Level` — 7 nhãn: `Intern`, `Fresher`, `Junior`, `Middle`, `Senior`, `Lead/Manager`, `Không rõ`):**
   - **Chặn 2 bẫy ngữ nghĩa tiếng Việt trên tiêu đề:**  
     (a) Không dùng từ `"quản lý"` đứng tự do cho `Lead/Manager` vì trong tiếng Việt cụm *"Chuyên viên / Kỹ sư quản lý [chất lượng dữ liệu / rủi ro / ứng dụng / dự án]"* là vị trí chuyên viên thực thi nghiệp vụ quản lý một đối tượng, không phải cấp quản lý con người. Chỉ gán `Lead/Manager` khi tiêu đề chứa chức danh lãnh đạo rõ ràng (`Giám đốc / Director`, `Trưởng phòng / Phó phòng / Trưởng bộ phận / Trưởng ban / Trưởng nhóm`, `Head of`, `Manager`, `Team Lead / Tech Lead / Leader / Lead`, `Chief / CTO / CIO / CDO / VP`).  
     (b) Không dùng từ `"chính"` đứng lẻ cho `Senior` (tránh bắt nhầm `"Tài chính"`, `"Hành chính"`); chỉ bắt `"chuyên viên chính"`, `"kỹ sư chính"`, các từ viết tắt cấp bậc ngân hàng (`CVCC` = Chuyên viên cao cấp, `CVC` = Chuyên viên chính) và các chức danh chuyên gia kỹ thuật cao cấp (`Senior`, `Chuyên gia`, `Expert`, `Architect`, `Kiến trúc sư`, `Principal`).
   - **Quy tắc Fallback từ `Exp_Min_Years`:** Với các tin không ghi cấp bậc trên tiêu đề, suy luận từ số năm kinh nghiệm tối thiểu (`< 1 năm` → `Fresher`, `1–2 năm` → `Junior`, `3–4 năm` → `Middle`, `>= 5 năm` → `Senior`). *Lưu ý cho Bước 2.5:* Nếu chọn bài toán phân lớp `Level`, bắt buộc loại bỏ các cột kinh nghiệm khỏi tập đặc trưng `X` để chống rò rỉ dữ liệu.

In [9]:
# [Mở rộng ngoài TH01-TH06] Cell 5A: Định nghĩa hàm trích xuất số năm kinh nghiệm và suy luận cấp bậc (Level)
NO_EXP_KEYWORDS: tuple[str, ...] = (
    "không yêu cầu", "chưa có kinh nghiệm", "không cần kinh nghiệm", "no experience", "mới tốt nghiệp",
)
UNDER_ONE_YEAR_KEYWORDS: tuple[str, ...] = ("dưới 1 năm", "less than 1 year", "< 1 năm", "dưới 1")


def parse_min_experience_years(exp_raw: object) -> float:
    '''Trích xuất số năm kinh nghiệm tối thiểu (Exp_Min_Years) từ chuỗi experience_raw.'''
    if pd.isna(exp_raw) or not str(exp_raw).strip():
        return np.nan
    low = str(exp_raw).strip().lower()

    if any(kw in low for kw in NO_EXP_KEYWORDS):
        return 0.0
    if any(kw in low for kw in UNDER_ONE_YEAR_KEYWORDS):
        return 0.5

    month_match = re.search(r"(\d+)\s*tháng", low)
    if month_match and "năm" not in low:
        years = round(int(month_match.group(1)) / 12.0, 2)
        return 0.5 if years < 1.0 else years

    num_matches = re.findall(r"\d+(?:[.,]\d+)?", low)
    if num_matches:
        return float(num_matches[0].replace(",", "."))
    return np.nan


INTERN_REGEX = re.compile(r"\b(?:intern|internship|thực\s*tập(?:\s*sinh)?|tts|cộng\s*tác\s*viên)\b", re.IGNORECASE)
FRESHER_REGEX = re.compile(r"\b(?:fresher|mới\s*tốt\s*nghiệp|graduate|trainee|tập\s*sự)\b", re.IGNORECASE)

# Chỉ bắt đúng chức danh lãnh đạo/quản lý con người (không bắt 'quản lý <đối tượng>' đi sau Chuyên viên/Kỹ sư)
LEAD_MGR_REGEX = re.compile(
    r"\b(?:director|head\s*of|vp|chief|cto|cio|cdo|giám\s*đốc|phó\s*giám\s*đốc|"
    r"trưởng\s*phòng|phó\s*phòng|trưởng\s*bộ\s*phận|trưởng\s*ban|trưởng\s*nhóm|trưởng\s*trung\s*tâm|"
    r"manager|team\s*lead|tech\s*lead|leader|lead|supervisor)\b",
    re.IGNORECASE,
)

# Bao gồm Senior, Chuyên gia, Architect/Principal và viết tắt cấp bậc ngân hàng (CVCC, CVC)
SENIOR_REGEX = re.compile(
    r"\b(?:senior|sr\.?|cao\s*cấp|cấp\s*cao|cvcc|chuyên\s*gia|expert|"
    r"chuyên\s*viên\s*chính|kỹ\s*sư\s*chính|cvc|principal|architect|kiến\s*trúc\s*sư)\b",
    re.IGNORECASE,
)
MIDDLE_REGEX = re.compile(r"\b(?:middle|mid[\s-]level|mid)\b", re.IGNORECASE)
JUNIOR_REGEX = re.compile(r"\b(?:junior|jr\.?|sơ\s*cấp)\b", re.IGNORECASE)


def infer_level_with_source(title: object, emp_raw: object, exp_min: float) -> tuple[str, str]:
    '''Trả về (Level, nguồn_suy_luận) trong đó nguồn là 'title_direct', 'exp_fallback', hoặc 'missing'.'''
    combined_title = f"{'' if pd.isna(title) else title} {'' if pd.isna(emp_raw) else emp_raw}"

    if INTERN_REGEX.search(combined_title):
        return ("Intern", "title_direct")
    if FRESHER_REGEX.search(combined_title):
        return ("Fresher", "title_direct")
    if LEAD_MGR_REGEX.search(combined_title):
        return ("Lead/Manager", "title_direct")
    if SENIOR_REGEX.search(combined_title):
        return ("Senior", "title_direct")
    if MIDDLE_REGEX.search(combined_title):
        return ("Middle", "title_direct")
    if JUNIOR_REGEX.search(combined_title):
        return ("Junior", "title_direct")

    if not pd.isna(exp_min):
        if exp_min < 1.0:
            return ("Fresher", "exp_fallback")
        if exp_min <= 2.0:
            return ("Junior", "exp_fallback")
        if exp_min <= 4.0:
            return ("Middle", "exp_fallback")
        return ("Senior", "exp_fallback")

    return ("Không rõ", "missing")

In [10]:
# Cell 5B: Áp dụng chuẩn hóa Kinh nghiệm, Cấp bậc và kiểm chứng các mẫu tiêu đề 'quản lý'
df_clean["Exp_Min_Years"] = df_clean["experience_raw"].map(parse_min_experience_years)

exp_cut = pd.cut(
    df_clean["Exp_Min_Years"],
    bins=[0.0, 0.9, 2.0, 5.0, np.inf],
    labels=["0 năm", "1-2 năm", "3-5 năm", ">5 năm"],
    include_lowest=True,
).astype("object")
df_clean["Exp_Group"] = exp_cut.fillna("Không rõ")

level_results = [
    infer_level_with_source(ttl, emp, exp_val)
    for ttl, emp, exp_val in zip(df_clean["title"], df_clean["employment_type_raw"], df_clean["Exp_Min_Years"])
]
df_clean["Level"] = [res[0] for res in level_results]
level_sources = pd.Series([res[1] for res in level_results], index=df_clean.index)

print("=== 1. PHÂN BỐ NHÓM KINH NGHIỆM (Exp_Group) ===")
exp_dist = df_clean["Exp_Group"].value_counts().to_frame(name="Số tin")
exp_dist["Tỷ lệ (%)"] = (exp_dist["Số tin"] / len(df_clean) * 100).round(2)
print(exp_dist)

print("\n=== 2. TỶ LỆ NGUỒN SUY LUẬN CẤP BẬC (Trực tiếp từ Tiêu đề vs Fallback Kinh nghiệm) ===")
src_dist = level_sources.value_counts().to_frame(name="Số tin")
src_dist["Tỷ lệ (%)"] = (src_dist["Số tin"] / len(df_clean) * 100).round(2)
print(src_dist)

print("\n=== 3. PHÂN BỐ CẤP BẬC CÔNG VIỆC (Level) ===")
level_order = ["Intern", "Fresher", "Junior", "Middle", "Senior", "Lead/Manager", "Không rõ"]
lvl_dist = df_clean["Level"].value_counts().reindex(level_order, fill_value=0).to_frame(name="Số tin")
lvl_dist["Tỷ lệ (%)"] = (lvl_dist["Số tin"] / len(df_clean) * 100).round(2)
print(lvl_dist)

print("\n=== 4. BẢNG CHÉO LEVEL x EXP_GROUP (Kiểm chứng tính nhất quán) ===")
exp_order = ["0 năm", "1-2 năm", "3-5 năm", ">5 năm", "Không rõ"]
ct_level_exp = pd.crosstab(df_clean["Level"], df_clean["Exp_Group"], margins=True).reindex(
    index=level_order + ["All"], columns=exp_order + ["All"], fill_value=0
)
print(ct_level_exp)

print("\n=== 5. KIỂM CHỨNG CÁC TIN CÓ CHỮ 'QUẢN LÝ' TRONG TIÊU ĐỀ (KHÔNG KÈM TRƯỞNG/GIÁM ĐỐC/MANAGER/LEAD) ===")
mask_ql_object = (
    df_clean["title"].str.contains(r"quản\s*lý", case=False, na=False)
    & ~df_clean["title"].str.contains(r"trưởng|giám\s*đốc|manager|head|lead|phó|director|chief|cto|cio|cdo|\bvp\b|supervisor", case=False, na=False)
)
print("Phân bố Level của nhóm 'Chuyên viên/Kỹ sư quản lý <đối tượng>':", df_clean.loc[mask_ql_object, "Level"].value_counts().to_dict())
print(df_clean.loc[mask_ql_object, ["title", "experience_raw", "Exp_Min_Years", "Level"]].head(8).to_string(index=False))

=== 1. PHÂN BỐ NHÓM KINH NGHIỆM (Exp_Group) ===
           Số tin  Tỷ lệ (%)
Exp_Group                   
3-5 năm       523      45.13
1-2 năm       442      38.14
0 năm          96       8.28
>5 năm         52       4.49
Không rõ       46       3.97

=== 2. TỶ LỆ NGUỒN SUY LUẬN CẤP BẬC (Trực tiếp từ Tiêu đề vs Fallback Kinh nghiệm) ===
              Số tin  Tỷ lệ (%)
exp_fallback     628      54.18
title_direct     502      43.31
missing           29       2.50

=== 3. PHÂN BỐ CẤP BẬC CÔNG VIỆC (Level) ===
              Số tin  Tỷ lệ (%)
Level                          
Intern            33       2.85
Fresher           55       4.75
Junior           337      29.08
Middle           221      19.07
Senior           293      25.28
Lead/Manager     191      16.48
Không rõ          29       2.50

=== 4. BẢNG CHÉO LEVEL x EXP_GROUP (Kiểm chứng tính nhất quán) ===
Exp_Group     0 năm  1-2 năm  3-5 năm  >5 năm  Không rõ   All
Level                                                        
Intern 

---
## 6. Làm sạch Văn bản, Loại bỏ Stop Words & Trích xuất Ma trận 73 Kỹ năng (`skill_dictionary.csv`)

**Mục tiêu & Giải trình thiết kế:**
1. **Nguồn văn bản chuẩn (`text_clean`) & Bảo toàn ký tự công nghệ:**
   - Gộp bắt buộc `title + description + requirements` (không dùng cột `skills_raw` của TopCV vì chứa chuỗi thẻ SEO dài trên `131.072` ký tự).
   - Quy trình làm sạch 3 bước: (a) Gỡ sạch thẻ HTML bằng `BeautifulSoup` và giải mã HTML entities; (b) Khớp 73 biểu thức chính quy từ `skill_dictionary.csv` trên văn bản đã gỡ HTML; (c) Hạ chữ thường, **loại bỏ các dấu câu/ký tự trang trí thừa** (`•`, `·`, `▪`, `,`, `;`, `:`, `!`, `?`, `"`, `(`, `)`) nhưng **chủ đích giữ lại 5 ký tự kỹ thuật** (`+`, `#`, `.`, `/`, `-`) để không làm biến dạng các thuật ngữ công nghệ (`c++`, `c#`, `.net`, `node.js`, `ci/cd`, `pl/sql`, `t-sql`, `scikit-learn`), sau đó xóa **Stop Words tiếng Việt và tiếng Anh** để lưu vào cột `text_clean`.
2. **Phân nhóm 73 kỹ năng vào 7 nhóm chuyên môn (`lang`, `framework`, `tool`, `cloud`, `db`, `domain`, `soft`):**
   - Mở rộng từ 5 nhóm gợi ý ban đầu bằng cách tách rõ nhóm công cụ BI/BA (`tool`: Excel, Power BI, Tableau, ERP/SAP, Jira, BPMN, Figma...) và nhóm kỹ thuật chuyên môn AI/Data/BA (`domain`: Machine Learning, LLM/RAG, ETL, Data Governance, BRD/SRS, Testing/UAT...) giúp nâng độ phủ kỹ năng chuyên môn lên trên **99%** và phân tách sắc nét các cụm nghề nghiệp ở Bước 2.3.
3. **Kiểm soát ranh giới từ chống khớp nhầm từ ngắn:**  
   Đặt `case_sensitive = 1` cho `AI` (`\bAI\b`) để loại bỏ 29 tin chứa đại từ tiếng Việt `"ai"`; dùng lookbehind `(?<!no)` cho `SQL` để tách khỏi `NoSQL`; dùng lookahead `(?!\s*script)` cho `Java`; và dùng ranh giới ký tự chữ-số cho `C#`, `C++`, `R`.

In [11]:
# [Mở rộng ngoài TH01-TH06] Cell 6A: Định nghĩa bộ Stop Words Việt/Anh và hàm làm sạch văn bản bảo toàn ký tự kỹ thuật
STOPWORDS_VI_EN: set[str] = {
    # Stop words tiếng Việt phổ biến trong JD
    "và", "của", "các", "có", "được", "cho", "với", "trong", "là", "để", "theo",
    "từ", "về", "như", "hoặc", "khi", "trên", "đến", "một", "những", "này", "đã",
    "sẽ", "không", "tại", "bằng", "qua", "ra", "vào", "cũng", "nhiều", "rất", "hơn",
    # Stop words tiếng Anh phổ biến trong JD
    "the", "and", "to", "of", "in", "for", "with", "on", "as", "are", "is", "be",
    "by", "at", "or", "from", "an", "that", "this", "will", "have", "has", "our",
    "you", "your", "we", "can", "into", "such", "their", "all", "but", "not",
}

STOPWORDS_REGEX = re.compile(
    r"\b(?:" + "|".join(re.escape(w) for w in sorted(STOPWORDS_VI_EN, key=len, reverse=True)) + r")\b",
    re.IGNORECASE,
)

# Xóa các dấu câu trang trí và ký tự bullet thừa nhưng giữ lại +, #, ., /, - cho thuật ngữ CNTT (c++, c#, .net, ci/cd)
DECORATIVE_PUNCT_REGEX = re.compile(r"[•·▪►♦✓✔★☆,;:\!\?\"'“”‘’\(\)\[\]\{\}<>|\\~`@$%^&*=_]+")


def clean_text_preserve_tech_tokens(text_lower: str) -> str:
    '''Xóa dấu câu trang trí thừa (giữ +, #, ., /, -), loại bỏ stop words Việt/Anh và chuẩn hóa khoảng trắng.'''
    no_punct = DECORATIVE_PUNCT_REGEX.sub(" ", text_lower)
    # Xóa dấu chấm hoặc gạch ngang đứng rời rạc ở cuối từ/câu (không xóa .net, node.js, c++, c#)
    no_trailing_dot = re.sub(r"(?<=\w)\.(?=\s|$)|(?:\s|^)-(?=\s|$)", " ", no_punct)
    no_stops = STOPWORDS_REGEX.sub(" ", no_trailing_dot)
    return re.sub(r"\s+", " ", no_stops).strip()


# 1. Văn bản đã gỡ HTML giữ nguyên chữ Hoa/Thường (khớp chính xác regex case_sensitive=1 như 'AI')
cased_clean_texts: list[str] = df_clean["_combined_text"].tolist()
lower_raw_texts: list[str] = df_clean["_combined_text"].str.lower().tolist()

# 2. Khớp 73 kỹ năng từ skill_dictionary.csv
skill_matrix_dict: dict[str, list[int]] = {}
skill_names_per_row: list[list[str]] = [[] for _ in range(len(df_clean))]

for cat, skill_name, col_name, rx in COMPILED_SKILLS:
    is_case_sensitive = (rx.flags & re.IGNORECASE) == 0
    target_texts = cased_clean_texts if is_case_sensitive else lower_raw_texts

    col_values: list[int] = []
    for idx, txt in enumerate(target_texts):
        matched = 1 if rx.search(txt) else 0
        col_values.append(matched)
        if matched:
            skill_names_per_row[idx].append(skill_name)
    skill_matrix_dict[col_name] = col_values

# 3. Tạo cột text_clean, skills_list, skill_count và ghép 73 cột nhị phân skill_*
df_clean["text_clean"] = [clean_text_preserve_tech_tokens(txt) for txt in lower_raw_texts]
df_clean["skills_list"] = [";".join(names) for names in skill_names_per_row]
df_clean["skill_count"] = [len(names) for names in skill_names_per_row]

df_skill_cols = pd.DataFrame(skill_matrix_dict, index=df_clean.index, dtype="int64")
df_clean = pd.concat([df_clean, df_skill_cols], axis=1)

In [12]:
# Cell 6B: Thống kê độ phủ theo 7 nhóm kỹ năng, Top 20 kỹ năng và kiểm tra mẫu chống khớp giả
print("=== 0. ĐỐI SOÁT VĂN BẢN TRƯỚC & SAU KHI LÀM SẠCH DẤU CÂU THỪA + STOP WORDS VIỆT/ANH ===")
print("  Trước khi xử lý:", lower_raw_texts[0][:145], "...")
print("  Sau khi xử lý  :", df_clean["text_clean"].iloc[0][:145], "...")

category_summary = []
for cat in ("lang", "framework", "tool", "cloud", "db", "domain", "soft"):
    cat_cols = [col for c, _, col, _ in COMPILED_SKILLS if c == cat]
    hit_jobs = int((df_clean[cat_cols].sum(axis=1) > 0).sum())
    category_summary.append({
        "Nhóm kỹ năng (category)": cat,
        "Số cột kỹ năng": len(cat_cols),
        "Số tin có >= 1 kỹ năng": hit_jobs,
        "Độ phủ (%)": round(hit_jobs / len(df_clean) * 100, 2),
        "Tổng lượt xuất hiện": int(df_clean[cat_cols].sum().sum()),
    })

df_cat_summary = pd.DataFrame(category_summary)
print("\n=== 1. ĐỘ PHỦ KỸ NĂNG THEO 7 NHÓM TRÊN TẬP DỮ LIỆU SẠCH ===")
print(df_cat_summary.to_string(index=False))

hard_cols = [col for c, _, col, _ in COMPILED_SKILLS if c != "soft"]
n_hard_covered = int((df_clean[hard_cols].sum(axis=1) > 0).sum())
n_any_covered = int((df_clean["skill_count"] > 0).sum())
print(f"\nĐộ phủ kỹ năng chuyên môn (6 nhóm Hard/Tool/Domain): {n_hard_covered}/{len(df_clean)} tin ({n_hard_covered/len(df_clean)*100:.2f}%)")
print(f"Độ phủ tổng (cả 7 nhóm gồm Soft skills)             : {n_any_covered}/{len(df_clean)} tin ({n_any_covered/len(df_clean)*100:.2f}%)")
print(f"Số kỹ năng trung bình / tin                         : {df_clean['skill_count'].mean():.2f} (Median = {df_clean['skill_count'].median():.1f})")

skill_freq_rows = []
for cat, skill_name, col_name, _ in COMPILED_SKILLS:
    cnt = int(df_clean[col_name].sum())
    skill_freq_rows.append({
        "Hạng": 0,
        "Nhóm": cat,
        "Tên kỹ năng": skill_name,
        "Tên cột": col_name,
        "Số tin": cnt,
        "Tỷ lệ (%)": round(cnt / len(df_clean) * 100, 2),
    })

df_skill_freq = pd.DataFrame(skill_freq_rows).sort_values(by="Số tin", ascending=False).reset_index(drop=True)
df_skill_freq["Hạng"] = df_skill_freq.index + 1
print("\n=== 2. TOP 20 KỸ NĂNG XUẤT HIỆN NHIỀU NHẤT ===")
print(df_skill_freq.head(20).to_string(index=False))

print("\n=== 3. KIỂM TRA MẪU NGỮ CẢNH KHỚP THẬT CỦA CÁC TỪ KHÓA NGẮN (AI, R, C#, C++, SQL) ===")
spot_check_targets = ["skill_domain_ai", "skill_lang_r", "skill_lang_csharp", "skill_lang_cpp", "skill_lang_sql"]
for target_col in spot_check_targets:
    row_meta = next(item for item in COMPILED_SKILLS if item[2] == target_col)
    _, s_name, c_name, rx = row_meta
    matched_texts = [cased_clean_texts[i] for i, val in enumerate(df_clean[c_name]) if val == 1]
    snippets = []
    for m_txt in matched_texts[:3]:
        m = rx.search(m_txt)
        if m:
            start = max(0, m.start() - 30)
            end = min(len(m_txt), m.end() + 35)
            snippets.append(m_txt[start:end])
    print(f"  [{s_name:28s} | {c_name} = {int(df_clean[c_name].sum()):3d} tin] Mẫu khớp: {snippets}")

=== 0. ĐỐI SOÁT VĂN BẢN TRƯỚC & SAU KHI LÀM SẠCH DẤU CÂU THỪA + STOP WORDS VIỆT/ANH ===
  Trước khi xử lý: it business analyst - làm việc với các phòng, ban và đơn vị vận hành để khảo sát, thu thập và phân tích nhu cầu sử dụng phần mềm. - phân tích quy ...
  Sau khi xử lý  : it business analyst làm việc phòng ban đơn vị vận hành khảo sát thu thập phân tích nhu cầu sử dụng phần mềm phân tích quy trình hiện xác định tồn ...

=== 1. ĐỘ PHỦ KỸ NĂNG THEO 7 NHÓM TRÊN TẬP DỮ LIỆU SẠCH ===
Nhóm kỹ năng (category)  Số cột kỹ năng  Số tin có >= 1 kỹ năng  Độ phủ (%)  Tổng lượt xuất hiện
                   lang               8                     608       52.46                 1095
              framework              10                     197       17.00                  362
                   tool              12                     598       51.60                 1265
                  cloud               9                     253       21.83                  664
                     db     

---
## 7. Kiểm định Toàn vẹn Dữ liệu (`assert`) & Xuất tệp `clean_data.csv`

**Mục tiêu kiểm định trước khi bàn giao cho Bước 2.2 → 2.5:**
1. Đúng **20 cột cố định** theo quy ước schema của nhóm + **73 cột nhị phân `skill_<nhóm>_<tên>`** = **93 cột**.
2. Toàn vẹn khóa truy vết: `job_id`, `site`, `URL_Goc`, `Timestamp` không khuyết, `job_id` là duy nhất.
3. Toàn vẹn logic lương: `Luong_Min <= Luong_Max` trên mọi dòng `range`; `Luong_Mid` chỉ tồn tại khi `salary_status == 'range'`; không có dòng `negotiable`/`missing` nào bị gán lương bằng `0`.
4. Toàn vẹn miền giá trị phân loại: `salary_status`, `Salary_Bracket`, `Location_Std`, `Exp_Group`, `Level` chỉ chứa đúng các nhãn chuẩn.
5. Toàn vẹn ma trận kỹ năng: 73 cột `skill_*` chỉ nhận giá trị nguyên `{0, 1}`, không có kỹ năng nào có tần suất bằng `0`, và `skill_count` bằng đúng tổng 73 cột `skill_*` trên từng dòng.

In [13]:
# 1. Định nghĩa đúng 20 cột cố định + 73 cột skill_*
FIXED_20_COLUMNS: list[str] = [
    "job_id", "site", "URL_Goc", "Timestamp",
    "title", "company",
    "salary_raw", "Luong_Min", "Luong_Max", "Luong_Mid", "salary_status", "Salary_Bracket",
    "Location_Std",
    "experience_raw", "Exp_Min_Years", "Exp_Group",
    "Level",
    "text_clean",
    "skills_list", "skill_count",
]
SKILL_73_COLUMNS: list[str] = [col for _, _, col, _ in COMPILED_SKILLS]
FINAL_COLUMNS: list[str] = FIXED_20_COLUMNS + SKILL_73_COLUMNS

df_final = df_clean[FINAL_COLUMNS].copy()

# 2. Chuỗi assert kiểm định toàn vẹn nghiêm ngặt
assert len(FIXED_20_COLUMNS) == 20, "Sai số lượng 20 cột cố định!"
assert len(SKILL_73_COLUMNS) == 73, "Sai số lượng 73 cột kỹ năng!"
assert df_final.shape[1] == 93, f"Tổng số cột phải là 93, thực tế: {df_final.shape[1]}"
assert len(df_final) >= 800, f"Quy mô dữ liệu sạch ({len(df_final)}) phải >= 800 tin theo yêu cầu đề bài!"

assert df_final["job_id"].isna().sum() == 0 and df_final["URL_Goc"].isna().sum() == 0, "Khóa truy vết bị khuyết!"
assert df_final["job_id"].is_unique, "Phát hiện trùng lặp job_id trong dữ liệu sạch!"
assert set(df_final["site"].unique()) == {"careerviet", "topcv", "vietnamworks"}, "Giá trị cột site không hợp lệ!"

assert set(df_final["salary_status"].unique()) <= {"range", "single_bound", "negotiable", "missing"}
mask_range = df_final["salary_status"].eq("range")
assert (df_final.loc[mask_range, "Luong_Min"] <= df_final.loc[mask_range, "Luong_Max"]).all(), "Lỗi Luong_Min > Luong_Max!"
assert df_final.loc[mask_range, "Luong_Mid"].notna().all(), "Dòng 'range' bị thiếu Luong_Mid!"
assert df_final.loc[~mask_range, "Luong_Mid"].isna().all(), "Luong_Mid chỉ được phép tính khi salary_status == 'range'!"
mask_neg_or_miss = df_final["salary_status"].isin(["negotiable", "missing"])
assert df_final.loc[mask_neg_or_miss, ["Luong_Min", "Luong_Max", "Luong_Mid"]].isna().all().all(), "Lỗi gán số cho lương thỏa thuận/khuyết!"
assert set(df_final["Salary_Bracket"].dropna().unique()) == {"Duoi_15M", "15M_30M", "Tren_30M"}

assert set(df_final["Location_Std"].unique()) <= {"Hà Nội", "TP.HCM", "Đà Nẵng", "Remote", "Đa địa điểm", "Tỉnh khác", "Không rõ"}
assert set(df_final["Exp_Group"].unique()) <= {"0 năm", "1-2 năm", "3-5 năm", ">5 năm", "Không rõ"}
assert set(df_final["Level"].unique()) <= {"Intern", "Fresher", "Junior", "Middle", "Senior", "Lead/Manager", "Không rõ"}

skill_vals = set(np.unique(df_final[SKILL_73_COLUMNS].values))
assert skill_vals == {0, 1}, f"Ma trận kỹ năng chứa giá trị ngoài {{0, 1}}: {skill_vals}"
assert (df_final[SKILL_73_COLUMNS].sum(axis=1) == df_final["skill_count"]).all(), "Tổng cột skill_* không khớp skill_count!"
assert (df_final[SKILL_73_COLUMNS].sum(axis=0) > 0).all(), "Có cột kỹ năng tần suất bằng 0!"

# 3. Xuất tệp clean_data.csv và đọc ngược lại từ ổ đĩa để xác nhận
df_final.to_csv(config.CLEAN_DATA_PATH, index=False, encoding=config.CSV_ENCODING)
df_reload = pd.read_csv(config.CLEAN_DATA_PATH, encoding=config.CSV_ENCODING)
assert df_reload.shape == df_final.shape, "Kích thước file đọc lại không khớp!"

print("=== KIỂM ĐỊNH TOÀN VẸN THÀNH CÔNG 100% (TẤT CẢ ASSERT ĐỀU PASS) ===")
print(f"Đường dẫn tệp sạch : {config.CLEAN_DATA_PATH} (tương đối: {config.CLEAN_DATA_REL_PATH})")
print(f"Kích thước tệp sạch: {df_reload.shape[0]:,} dòng x {df_reload.shape[1]} cột (20 cột cố định + 73 cột skill_*)")
print(f"Dung lượng tệp     : {config.CLEAN_DATA_PATH.stat().st_size / (1024 * 1024):.2f} MB")
print("\n--- Xem trước 5 dòng đầu tiên của clean_data.csv (14 cột thông tin chính) ---")
summary_cols = [
    "job_id", "site", "title", "Luong_Min", "Luong_Max", "Luong_Mid",
    "salary_status", "Salary_Bracket", "Location_Std", "Exp_Min_Years",
    "Exp_Group", "Level", "skill_count", "skills_list",
]
print(df_reload[summary_cols].head(5).to_string(index=False))

=== KIỂM ĐỊNH TOÀN VẸN THÀNH CÔNG 100% (TẤT CẢ ASSERT ĐỀU PASS) ===
Đường dẫn tệp sạch : C:\Users\letrungtanh\Videos\NVIDIA\Gemini\Khai Phá Dữ Liệu\giai_doan_2.1\data\clean\clean_data.csv (tương đối: data/clean/clean_data.csv)
Kích thước tệp sạch: 1,159 dòng x 93 cột (20 cột cố định + 73 cột skill_*)
Dung lượng tệp     : 3.57 MB

--- Xem trước 5 dòng đầu tiên của clean_data.csv (14 cột thông tin chính) ---
             job_id       site                                          title  Luong_Min  Luong_Max  Luong_Mid salary_status Salary_Bracket Location_Std  Exp_Min_Years Exp_Group        Level  skill_count                                                                                                                                                                                                                                                              skills_list
careerviet:35C8B6A1 careerviet                            IT Business Analyst       18.0       30.0       24.0         ra

---
## Tổng kết Bước 2.1

### Q&A
- **Câu hỏi 1: Hàm Regex bóc tách mức lương xử lý sự khác biệt dấu `.` và `,` giữa CareerViet với TopCV/VietnamWorks như thế nào?**  
  Thay vì chia cứng theo tên sàn dễ lỗi khi dữ liệu hỗn hợp, biểu thức `THOUSANDS_SEP_REGEX` dùng positive lookahead `(?=\d{3}(?:[.,]\d{3})*(?:\D|$))` để nhận diện mọi dấu `.` hoặc `,` đứng trước đúng cụm 3 chữ số (ví dụ `1.000`, `2,500`, `18.000.000`) là dấu phân cách hàng nghìn và loại bỏ chúng; mọi dấu `,` còn lại đứng trước 1–2 chữ số (ví dụ `15,5`) là dấu thập phân và được đổi thành `.`. Sau đó hàm nhân với tỷ giá trung tâm NHNN cố định ngày 19/11/2026 (`1 USD = 25.646 VNĐ`) nếu đơn vị là USD và chia `12` nếu trả theo `/năm`.
- **Câu hỏi 2: Tại sao không điền `0` hoặc `Mean/Median` vào các ô lương "Thỏa thuận"?**  
  Trong 1,159 tin sạch có tới **802 tin (69.20%)** ghi lương thỏa thuận. Điền `0` sẽ kéo sập mức lương trung bình về sát 0 và tạo giá trị giả; điền `Mean/Median` của 26.23% tin công khai cho gần 70% tin còn lại sẽ làm phương sai sụp đổ và tạo đỉnh giả tạo, gây sai lệch nghiêm trọng cho phân tích EDA (Bước 2.2) cũng như làm mô hình phân lớp (Bước 2.5) học trên nhãn nhân tạo. Do đó ta giữ nguyên `NaN` và quản lý qua cột `salary_status`.
- **Câu hỏi 3: Tại sao cần lọc tin lạc ngành và khử trùng theo cặp `(title, company)` ngoài `job_id`?**  
  Khóa `job_id` chỉ là mã nội bộ riêng của từng sàn; một công ty thường đăng cùng một tin tuyển dụng lên cả CareerViet, TopCV và VietnamWorks hoặc đăng lại nhiều lần (phát hiện **27 bản ghi trùng** `(title, company)`). Ngoài ra, cơ chế tìm kiếm từ khóa của sàn trả về **91 tin** lỗi `topcv + Tableau` và **196 tin** ngoài ngành CNTT/Dữ liệu/BA. Loại bỏ chúng giúp độ phủ kỹ năng chuyên môn trên tập sạch đạt tới **98.02%**.
- **Câu hỏi 4: Làm thế nào để trích xuất các kỹ năng có tên ngắn như `AI`, `R`, `C#`, `C++`, `SQL` mà không bắt nhầm vào giữa từ tiếng Việt/Anh khác?**  
  Từ điển `skill_dictionary.csv` thiết lập `case_sensitive = 1` cho `AI` (`\bAI\b`) giúp loại bỏ 29 tin khớp nhầm chữ `"ai"` tiếng Việt; dùng lookbehind `(?<!no)` cho `SQL` để không đếm nhầm `NoSQL`; dùng lookahead `(?!\s*script)` cho `Java` để không đếm nhầm `JavaScript`; dùng ranh giới ký tự chữ-số `(?<![a-z0-9])` cho `C#` và `C++` (do `#` và `+` không thuộc `\w`); và kiểm soát ngữ cảnh danh sách ngôn ngữ lân cận cho `R`.

### Data Analysis Key Findings
- **Làm sạch & Khử trùng (Cleaning Audit Log):** Từ **1,473 tin thô**, quy trình loại bỏ **91 tin** lỗi tìm kiếm `topcv + Tableau`, **196 tin** lạc ngành (kế toán, y tế, cơ khí, marketing, sales, phân tích đầu tư và tin không chứa kỹ năng chuyên môn) và **27 tin** trùng lặp `(title, company)`, thu được **1,159 tin tuyển dụng sạch (78.68%)** phân bố trên `careerviet` (**574 tin**), `topcv` (**306 tin**) và `vietnamworks` (**279 tin**).
- **Bức tranh Mức lương chuẩn hóa:** Có **304 tin — 26.23%** công bố khoảng lương đủ 2 cận (`range`) với mức lương trung bình `Luong_Mid` đạt **Mean = 24.79 triệu VNĐ/tháng** (Median = **22.50 triệu VNĐ/tháng**, Min = **1.50 triệu**, Max = **85.00 triệu**), chia thành 3 khoảng `15M_30M` (**179 tin**), `Tren_30M` (**71 tin**) và `Duoi_15M` (**54 tin**); **48 tin — 4.14%** công bố 1 cận (`single_bound`); **802 tin — 69.20%** lương thỏa thuận (`negotiable`); và **5 tin — 0.43%** khuyết/lỗi placeholder (`missing`).
- **Địa điểm, Kinh nghiệm & Cấp bậc:** Nhu cầu tuyển dụng tập trung tại **Hà Nội (638 tin — 55.05%)** và **TP.HCM (361 tin — 31.15%)**, tiếp theo là **Tỉnh khác (118 tin — 10.18%)**, **Đa địa điểm (22 tin — 1.90%)**, **Đà Nẵng (13 tin — 1.12%)**, **Remote (4 tin — 0.35%)** và **Không rõ (3 tin — 0.26%)**. Nhóm kinh nghiệm phổ biến nhất là **3–5 năm (523 tin — 45.13%)** và **1–2 năm (442 tin — 38.14%)**, tiếp theo là **0 năm (96 tin — 8.28%)**, **>5 năm (52 tin — 4.49%)** và **Không rõ (46 tin — 3.97%)**. Phân bố cấp bậc sau khi chuẩn hóa dẫn đầu bởi **Junior (337 tin — 29.08%)**, **Senior (293 tin — 25.28%)**, **Middle (221 tin — 19.07%)**, **Lead/Manager (191 tin — 16.48%)**, **Fresher (55 tin — 4.75%)**, **Intern (33 tin — 2.85%)** và **Không rõ (29 tin — 2.50%)**.
- **Ma trận 73 Kỹ năng thuộc 7 Nhóm kỹ năng:** Đạt độ phủ **98.02% (1136/1159 tin)** có ít nhất 1 kỹ năng chuyên môn và **99.22% (1150/1159 tin)** trên toàn bộ 7 nhóm kỹ năng, trung bình mỗi tin yêu cầu **7.67 kỹ năng** (Median = **7.0**). Top 5 kỹ năng chuyên môn dẫn đầu thị trường là **SQL (450 tin — 38.83%)**, **Artificial Intelligence (AI) (424 tin — 36.58%)**, **Python (341 tin — 29.42%)**, **Power BI (316 tin — 27.26%)**, **Excel (295 tin — 25.45%)**.

### Insights or Next Steps
- **Sẵn sàng cho Bước 2.2 (EDA) & Bước 2.3–2.4 (Gom cụm & Luật kết hợp):** Tệp `data/clean/clean_data.csv` có 1136/1159 bản ghi (98.02%) sở hữu kỹ năng chuyên môn (`> 0`), đảm bảo mật độ đặc trưng rất cao để phân tích đa chiều ở Bước 2.2, tìm số cụm tối ưu $K$ bằng Elbow/Silhouette ở Bước 2.3 và khai phá tập luật Apriori/FP-Growth ở Bước 2.4.
- **Lưu ý chống Rò rỉ Dữ liệu (Data Leakage) khi chuyển sang Bước 2.5:** Khi huấn luyện mô hình phân lớp cho `Salary_Bracket` (trên 304 tin `range`), bắt buộc loại bỏ `salary_raw`, `Luong_Min`, `Luong_Max`, `Luong_Mid`, `salary_status`; nếu phân lớp cho `Level`, bắt buộc loại bỏ `experience_raw`, `Exp_Min_Years`, `Exp_Group` khỏi tập đặc trưng đầu vào `X`.